### SETUP MODULES
The following modules include imports / IBM account initialization that the homework requires.

In [1]:
# -----------------------------------
# IMPORTS REQUIRED FOR THE ASSIGNMENT
# -----------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService

In [2]:
# -------------------------------
# ACCOUNT LINK TO QISKIT SERVVICE
# -------------------------------
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="hPp27YYsQIgcnApUBSMsQnMIBL-VkUwIHN5-aOHeKnkF",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/333c753dd333451a9f49f7993cede44f:95cf404b-9d0a-49c0-9690-bc0a1b6fc509::",
    overwrite=True,
    set_as_default=True
)

service = QiskitRuntimeService()
print("Connected to IBM Quantum.")

Connected to IBM Quantum.


### PART 1
In this part, I have:

- Chosen 2 available backends
- Read / store all properties from each qubit
- Find the min, max, median and mean for each property

### PART 1
In this part, I have:

- Chosen 2 available backends
- Read / store all properties from each qubit
- Find the min, max, median and mean for each property

In [3]:
# -------------------------------
# INITIALZING / CHOOSING BACKENDS
# -------------------------------
available_backends = service.backends(
    simulator=False,
    operational=True
)

# Viewing all potential backends
for backend in available_backends:
    print(backend)

# Selecting backend here
backend_one = service.backend('ibm_fez')
backend_two = service.backend('ibm_marrakesh')

<IBMBackend('ibm_fez')>
<IBMBackend('ibm_marrakesh')>
<IBMBackend('ibm_kingston')>


In [54]:
# ---------------------------------
# READING BACKEND CHARACTERIZATIONS
# ---------------------------------
device_info = {
    "ibm_fez": {"T1": [], "T2": [], "Readout": []},
    "ibm_marrakesh": {"T1": [], "T2": [], "Readout": []},
}

num_qubits_b1 = backend_one.num_qubits
num_qubits_b2 = backend_two.num_qubits

backend_one_prop = backend_one.properties(refresh=True)
backend_two_prop = backend_two.properties(refresh=True)

# Reading error from IBM_FEZ
for qubit in range(num_qubits_b1):
    qubit_properties = backend_one_prop.qubit_property(qubit)

    if "T1" in qubit_properties:
        device_info["ibm_fez"]["T1"].append(qubit_properties["T1"][0] * 1e6)
    else:
        device_info["ibm_fez"]["T1"].append(np.nan)

    if "T2" in qubit_properties:
        device_info["ibm_fez"]["T2"].append(qubit_properties["T2"][0] * 1e6)
    else:
        device_info["ibm_fez"]["T2"].append(np.nan)

    if "readout_error" in qubit_properties:
        device_info["ibm_fez"]["Readout"].append(qubit_properties["readout_error"][0])
    else:
        device_info["ibm_fez"]["Readout"].append(np.nan)

# Reading error from IBM_MERRAKESH
for qubit in range(num_qubits_b2):
    qubit_properties = backend_two_prop.qubit_property(qubit)

    if "T1" in qubit_properties:
        device_info["ibm_marrakesh"]["T1"].append(qubit_properties["T1"][0] * 1e6)
    else:
        device_info["ibm_marrakesh"]["T1"].append(np.nan)

    if "T2" in qubit_properties:
        device_info["ibm_marrakesh"]["T2"].append(qubit_properties["T2"][0] * 1e6)
    else:
        device_info["ibm_marrakesh"]["T2"].append(np.nan)

    if "readout_error" in qubit_properties:
        device_info["ibm_marrakesh"]["Readout"].append(qubit_properties["readout_error"][0])
    else:
        device_info["ibm_marrakesh"]["Readout"].append(np.nan)
   
fez_df = pd.DataFrame(device_info["ibm_fez"])
merrakesh_df = pd.DataFrame(device_info["ibm_marrakesh"])

# display(fez_df.style.set_caption("IBM FEZ"))
# display(merrakesh_df.style.set_caption("IBM MARRAKESH"))

In [56]:
# ---------------------------------
# READING GATE ERRORS
# ---------------------------------
target_fez = backend_one.target
target_marrakesh = backend_two.target

device_info["ibm_fez"]["SingleQubitGateErrors"] = {}
device_info["ibm_fez"]["TwoQubitGateErrors"] = {}
device_info["ibm_marrakesh"]["SingleQubitGateErrors"] = {}
device_info["ibm_marrakesh"]["TwoQubitGateErrors"] = {}

for name, target in [("ibm_fez", target_fez), ("ibm_marrakesh", target_marrakesh)]:
    for gate_name in target.operation_names:
        qubit_props = target[gate_name]
        for qubits, props in qubit_props.items():
            if qubits is None:
                continue  # globally-defined operation, not tied to a specific qubit/link

            if props is None or props.error is None:
                error = np.nan
            else:
                error = props.error

            if len(qubits) == 1:
                device_info[name]["SingleQubitGateErrors"].setdefault(gate_name, []).append(error)
            elif len(qubits) == 2:
                device_info[name]["TwoQubitGateErrors"].setdefault(gate_name, []).append(error)

In [57]:
# ----------------------------------
# SUMMARIZING MIN, MAX, MEAN, MEDIAN
# ----------------------------------

# Helper function to summarize properties of a backend
def summarize(values):
    np_array = np.array(values)
    return {
        "min": np.nanmin(np_array),
        "max": np.nanmax(np_array),
        "mean": np.nanmean(np_array),
        "median": np.nanmedian(np_array),
    }

fez_summary = {
    "T1": summarize(device_info["ibm_fez"]["T1"]),
    "T2": summarize(device_info["ibm_fez"]["T2"]),
    "Readout": summarize(device_info["ibm_fez"]["Readout"]),
    "SX Error": summarize(device_info["ibm_fez"]["SingleQubitGateErrors"]["sx"]),
    "X Error": summarize(device_info["ibm_fez"]["SingleQubitGateErrors"]["x"]),
    "CZ Error": summarize(device_info["ibm_fez"]["TwoQubitGateErrors"]["cz"]),
}

marrakesh_summary = {
    "T1": summarize(device_info["ibm_marrakesh"]["T1"]),
    "T2": summarize(device_info["ibm_marrakesh"]["T2"]),
    "Readout": summarize(device_info["ibm_marrakesh"]["Readout"]),
    "SX Error": summarize(device_info["ibm_marrakesh"]["SingleQubitGateErrors"]["sx"]),
    "X Error": summarize(device_info["ibm_marrakesh"]["SingleQubitGateErrors"]["x"]),
    "CZ Error": summarize(device_info["ibm_marrakesh"]["TwoQubitGateErrors"]["cz"]),
}

fez_summary_df = pd.DataFrame(fez_summary) 
marrakesh_summary_df = pd.DataFrame(marrakesh_summary)

display(fez_summary_df.style.set_caption("IBM FEZ SUMMARY"))
display(marrakesh_summary_df.style.set_caption("IBM MARRAKESH SUMMARY"))

,T1,T2,Readout,SX Error,X Error,CZ Error
min,17.761524,5.443194,0.003052,0.000123,0.000123,0.001154
max,331.350660,261.866226,0.316162,1.000000,1.000000,1.000000
mean,129.754101,95.989437,0.021024,0.006846,0.006846,0.033638
median,128.361346,90.712911,0.009216,0.000318,0.000318,0.002850


,T1,T2,Readout,SX Error,X Error,CZ Error
min,7.631817,5.831653,0.001465,0.000102,0.000102,0.001140
max,356.287697,433.491383,0.499023,1.000000,1.000000,1.000000
mean,166.632551,93.762165,0.033527,0.019761,0.019761,0.039721
median,157.076093,64.368522,0.011230,0.000378,0.000378,0.003112
